# Algorithms 21: Hash Tables**Learning Objectives:**1. Understand how Dictionaries/Maps are implemented under the hood2. Design a Hash Function3. Implement a Hash Table with Buckets to handle collisions**Prerequisites:** Algorithms 01**Estimated time:** 60 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 21

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "21_hash_tables",    "level": 4,    "title": "Algorithms 21: Hash Tables",    "prerequisites": [        "Algorithms 01"    ],    "skills_taught": [        "lesson.algorithms_21_hash_tables"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "22_simplex_method.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_21_hash_tables",        "title": "Lesson Algorithms 21 Hash Tables",        "notebook": "21_hash_tables.ipynb",        "level": 4    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "21_hash_tables.ipynb",        "level": 4    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 01.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_21_hash_tables', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### Dictionaries are Hash Tables> 📖 *From Algorithms Ch. 21:* Under the hood, dictionaries are hash tables. The most elementary (and inefficient) version of a dictionary is just a list of `(key, value)` tuples.If we have 1 million tuples, looking up a key requires $O(N)$ time. We must scan the whole list.### The Solution: Buckets and Hash FunctionsInstead of 1 giant list, what if we have 100 separate lists ("buckets")? We use a **Hash Function** that takes the string key, does some math on it, and spits out an integer from $0$ to $99$. Now, when we insert `("apple", 5)`, the hash function says `hash("apple") = 42`. We instantly go to bucket 42 and append it there.When we want to look up `"apple"`, we hash it, get 42, go to bucket 42, and search that tiny list. Time complexity drops to $O(1)$!

### Comprehension Check ✅1. What is a "Collision" in a hash table?2. What happens if our Hash Function is terribly designed and always outputs `0` regardless of the string?<details><summary>💡 Explanation after a retrieval attempt</summary>1. A collision happens when two completely different keys (e.g. "apple" and "banana") happen to yield the exact same hash integer, and thus end up in the exact same bucket.2. Every single key gets put into Bucket 0. The other 99 buckets remain empty. You essentially regress back to a single giant list, and performance drops to $O(N)$. This is why uniform hashing is critical!</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: A Simple Hash FunctionLet our hash function take a string, sum up the alphabetical index of each character (`A`=1, `B`=2... ignore case, skip non-letters), and then take the result modulo `num_buckets`.Write this function.

In [ ]:
def simple_hash(key_string, num_buckets):    total = 0    for char in key_string.lower():        if 'a' <= char <= 'z':            total += ord(char) - ord('a') + 1    return total % num_buckets# assert simple_hash("cab", 10) == 6 # (3 + 1 + 2) % 10 = 6# print('Phase 0 passed ✅')

---## Phase 1 — Algorithm Construction### Step 1: The Hash Table CoreImplement a `HashTable` class. It initializes `self.buckets` as a list of empty lists. (e.g., `[ [], [], [], ... ]`).

In [ ]:
class HashTable:    def __init__(self, num_buckets):        self.num_buckets = num_buckets        # Create a list containing `num_buckets` empty lists.        # IMPORTANT: Use list comprehension! self.buckets = [[]] * num_buckets creates references to the SAME list!        self.buckets = [[] for _ in range(num_buckets)]            def get_hash(self, key):        return simple_hash(key, self.num_buckets)

### Step 2: Insert / UpdateImplement the `put(key, value)` method.1. Hash the key to find the correct bucket.2. Scan the bucket. If the key already exists, *update* its value.3. If it doesn't exist, append the `(key, value)` tuple to the bucket.

In [ ]:
# Add this to your HashTable class:# def put(self, key, value):#     bucket_idx = self.get_hash(key)#     bucket = self.buckets[bucket_idx]#     #     # Check if key exists in bucket#     for i in range(len(bucket)):#         if bucket[i][0] == key:#             bucket[i] = (key, value) # Tuples are immutable, must replace#             return#             #     # If not found, append#     bucket.append((key, value))

### Step 3: RetrievalImplement the `get(key)` method. Return the value if found, or `None` if not found.

In [ ]:
# Add this to your HashTable class:# def get(self, key):#     bucket_idx = self.get_hash(key)#     # Scan the bucket for the key and return value#     # YOUR CODE HERE#     return None

---## Phase 2 — Experimental Verification### Testing Collisions"cab" and "abc" have the same letters, so our `simple_hash` will give them the exact same hash (Collision!). Let's make sure our Hash Table handles this gracefully.

In [ ]:
# table = HashTable(10)# table.put("cab", 5)# table.put("abc", 9)# print("Value for cab:", table.get("cab")) # Should be 5# print("Value for abc:", table.get("abc")) # Should be 9# print("Bucket array:", table.buckets) # Notice that the bucket at index 6 contains BOTH tuples!

---## Phase 3 — Olympiad Extension### Challenge: Dynamic ResizingIf we put 10,000 items into a HashTable with 10 buckets, each bucket has 1,000 items, destroying performance.Real dictionaries **resize** themselves. If the ratio of items to buckets exceeds a `load_factor` (e.g. 0.75), they create a new bucket array twice the size, and *re-hash* every single item into the new array.Write a `resize()` method!

In [ ]:
# YOUR CODE HEREpass

### Chalkboard DefenseExplain why Python's built-in `hash()` function for strings is randomized per session (a security feature). What "attack" does this prevent?<details><summary>💡 Sketch after a retrieval attempt</summary>If the hash algorithm is deterministic and public, a malicious user could intentionally generate 100,000 different string keys that *all* collide to Bucket 0. If they send a JSON payload with these keys to a web server, the server's Hash Table degrades to $O(N^2)$ insertion time, causing the server's CPU to max out and crash. This is a "Hash Collision Denial of Service (DoS)" attack! Randomizing the hash seed prevents attackers from pre-computing collisions.</details>---📚 **Next:** Algorithms 22 (Simplex Method)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_21_hash_tables', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='22_simplex_method.ipynb')